# Algoritmos Genéticos: Clase Individuo y Clase Población
**Referencia:** Kuri, Á. (2002). *Algoritmos genéticos*. Instituto Politécnico Nacional.

---
## Fundamento biológico
En la naturaleza, las especies evolucionan mediante la selección de los individuos más aptos.
Un **individuo** posee un **cromosoma** (cadena de genes) que codifica una solución al problema.
Una **población** es el conjunto de individuos que compiten, se reproducen y evolucionan.


In [ ]:
import random

class Individuo:
    """
    Representa un individuo dentro de una población en un Algoritmo Genético.

    En la analogía biológica (Kuri, 2002):
    - El cromosoma codifica una posible solución al problema.
    - La aptitud (fitness) mide qué tan buena es esa solución.
    - La mutación introduce variabilidad genética aleatoria.
    """

    def __init__(self, longitud_cromosoma: int, generacion: int = 0):
        """
        Inicializa un individuo con un cromosoma de longitud dada.

        Parámetros:
            longitud_cromosoma (int): Número de genes en el cromosoma.
            generacion (int): Generación en la que nació el individuo.
        """
        self.longitud_cromosoma = longitud_cromosoma
        self.generacion = generacion
        self.cromosoma = []      # Lista de genes (bits: 0 o 1)
        self.aptitud = 0.0       # Valor de aptitud (fitness)
        self.inicializar()

    def inicializar(self):
        """Genera un cromosoma aleatorio de bits (0 o 1)."""
        self.cromosoma = [random.randint(0, 1) for _ in range(self.longitud_cromosoma)]

    def calcular_aptitud(self):
        """
        Calcula la aptitud del individuo.
        En este ejemplo, la aptitud es la suma de unos en el cromosoma
        (problema OneMax: maximizar la cantidad de genes = 1).
        """
        self.aptitud = sum(self.cromosoma)
        return self.aptitud

    def mutar(self, prob_mutacion: float = 0.01):
        """
        Aplica mutación bit a bit según una probabilidad dada.

        Parámetros:
            prob_mutacion (float): Probabilidad de invertir cada gen (0.0 a 1.0).
        """
        for i in range(self.longitud_cromosoma):
            if random.random() < prob_mutacion:
                self.cromosoma[i] = 1 - self.cromosoma[i]  # Invierte el bit

    def obtener_gen(self, indice: int) -> int:
        """Retorna el valor del gen en la posición indicada."""
        return self.cromosoma[indice]

    def __repr__(self) -> str:
        genes = ''.join(map(str, self.cromosoma))
        return f"Individuo(gen={self.generacion}, cromosoma={genes}, aptitud={self.aptitud})"


In [ ]:
class Poblacion:
    """
    Representa la población de individuos en un Algoritmo Genético.

    En la analogía biológica (Kuri, 2002):
    - La población evoluciona generación a generación.
    - Se seleccionan los individuos más aptos para reproducirse.
    - El cruzamiento combina cromosomas de dos padres.
    - La mutación mantiene diversidad genética.
    """

    def __init__(self, tamanio: int, longitud_cromosoma: int, tasa_mutacion: float = 0.01):
        """
        Inicializa la población.

        Parámetros:
            tamanio (int): Número de individuos en la población.
            longitud_cromosoma (int): Longitud del cromosoma de cada individuo.
            tasa_mutacion (float): Probabilidad de mutación por gen.
        """
        self.tamanio = tamanio
        self.tasa_mutacion = tasa_mutacion
        self.generacion_actual = 0
        self.individuos = []
        self._longitud_cromosoma = longitud_cromosoma
        self.inicializar_poblacion()

    def inicializar_poblacion(self):
        """Crea la población inicial con individuos aleatorios."""
        self.individuos = [
            Individuo(self._longitud_cromosoma, generacion=0)
            for _ in range(self.tamanio)
        ]
        for ind in self.individuos:
            ind.calcular_aptitud()

    def seleccion(self) -> 'Individuo':
        """
        Selección por torneo: elige el mejor de 3 individuos al azar.
        Simula la presión selectiva de la naturaleza.

        Retorna:
            Individuo: El ganador del torneo.
        """
        competidores = random.sample(self.individuos, min(3, len(self.individuos)))
        return max(competidores, key=lambda ind: ind.aptitud)

    def cruzar(self, padre1: 'Individuo', padre2: 'Individuo') -> 'Individuo':
        """
        Cruzamiento de un punto: combina genes de dos padres.

        Parámetros:
            padre1, padre2 (Individuo): Padres seleccionados.

        Retorna:
            Individuo: Hijo resultante del cruzamiento.
        """
        punto = random.randint(1, self._longitud_cromosoma - 1)
        hijo = Individuo(self._longitud_cromosoma, generacion=self.generacion_actual + 1)
        hijo.cromosoma = padre1.cromosoma[:punto] + padre2.cromosoma[punto:]
        return hijo

    def evolucionar(self):
        """
        Produce una nueva generación mediante selección, cruzamiento y mutación.
        Reemplaza la población actual con los nuevos individuos.
        """
        nueva_generacion = []
        for _ in range(self.tamanio):
            padre1 = self.seleccion()
            padre2 = self.seleccion()
            hijo = self.cruzar(padre1, padre2)
            hijo.mutar(self.tasa_mutacion)
            hijo.calcular_aptitud()
            nueva_generacion.append(hijo)
        self.individuos = nueva_generacion
        self.generacion_actual += 1

    def mejor_individuo(self) -> 'Individuo':
        """Retorna el individuo con mayor aptitud en la generación actual."""
        return max(self.individuos, key=lambda ind: ind.aptitud)

    def __repr__(self) -> str:
        mejor = self.mejor_individuo()
        return (
            f"Población(generación={self.generacion_actual}, "
            f"tamaño={self.tamanio}, mejor_aptitud={mejor.aptitud})"
        )


## Prueba de las clases
Ejecutamos el algoritmo genético durante 10 generaciones para el problema **OneMax**.

In [ ]:
# Parámetros del algoritmo
TAMANIO_POBLACION = 20
LONGITUD_CROMOSOMA = 10
TASA_MUTACION = 0.05
GENERACIONES = 10

# Crear y evolucionar la población
poblacion = Poblacion(TAMANIO_POBLACION, LONGITUD_CROMOSOMA, TASA_MUTACION)
print("=== Evolución del Algoritmo Genético ===")
print(f"Generación 0: {poblacion}")

for g in range(GENERACIONES):
    poblacion.evolucionar()
    print(f"Generación {poblacion.generacion_actual}: {poblacion}")

print("\n=== Mejor individuo final ===")
print(poblacion.mejor_individuo())
